In [1]:
import xgi
import numpy as np
from tqdm import tqdm
import pickle
from multiprocess import Pool
from had_model import HAD_model
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Temporal evolution of hyperedges

Here we study the temporal evolution of:
- relative number of hyperedges (w.r.t. the initial number)
- average hyperedge size (± standard deviation)
- maximum hyperedge size

We fix some parameters of the starting structure: ER hypergraph, $M=4,\ \langle k \rangle = 10$, condition for agreement = max-min.

In [4]:
eps = 0.4
cond = 'max_min'   # condition for agreement
k_avg = 10
M = 4
n_runs = 50    # number of n_runs

# auxiliary functions to parallelize computations
def get_temporal_results(N):
        
    # All below quantities are computed along time, then averaged over 'n_runs'
    nhe = []      # relative number of hyperedges
    s_avg = []    # average hyperedge size
    s_max = []    # max hyperedge size
    s_std = []    # std of hyperedge sizes
    n_agree = []
    n_merge = []
    n_split = []
    
    for it in tqdm(range(n_runs)):

        nhe_it, s_avg_it = [], []   
        s_max_it, s_std_it = [], []
        
        H = xgi.uniform_erdos_renyi_hypergraph(N, M, k_avg, p_type="degree")
        H.cleanup()
        groups_0 = H.edges.members()
        
        Model = HAD_model(eps, groups_0)
        results = Model.simulate(condition=cond)
        opinions = results['opinions']
        groups = results['groups']
        n_events = results['n_events']
        T = len(groups)
        
        # get results over time t
        for t in range(T):
            sizes_t = [len(he) for he in groups[t]]
            s_avg_it.append( np.average(sizes_t) )
            s_max_it.append( max(sizes_t) )
            s_std_it.append( np.std(sizes_t) )
            nhe_it.append( len(groups[t]) / len(groups_0) )
        
        nhe.append(nhe_it)
        s_avg.append(s_avg_it)
        s_max.append(s_max_it)
        s_std.append(s_std_it)
        n_agree.append(n_events['agree'])
        n_split.append(n_events['split'])
        n_merge.append(n_events['merge'])

    # maximum simulation time among all the runs
    T_max = max([len(x) for x in nhe])
    res = []
    for measure in [nhe, s_avg, s_max, s_std, n_agree, n_split, n_merge]:
        # pad list of lists with last values at steady state, 
        # to obtain a rectangular matrix of shape (n_runs, T_max)
        measure = [x + [x[-1]] * (T_max-len(x)) for x in measure]
        # take the column-wise average over all the n_runs
        res.append( np.average(measure, axis=0) )
    
    results = {
        'nhe': res[0],
        's_avg': res[1],
        's_max': res[2],
        's_std': res[3],
        'n_agree': res[4],
        'n_split': res[5],
        'n_merge': res[6],
        'n_runs': n_runs,
        'k_avg': k_avg
        }
    
    return results

In [8]:
Ns = [500, 1000, 2000, 5000]

p = Pool(processes=4)
results_ = p.map(get_temporal_results, Ns)

# save results
results = {Ns[i]: results_[i] for i in range(len(Ns))}

with open(f'../results/ER_{cond}/ER_M{M}_{cond}_epsilon_{eps}_temporal.pkl', 'wb') as fp:
        pickle.dump(results, fp)

100%|████████████████████████████████████████| 50/50 [1:43:26<00:00, 124.14s/it]


# Single run to check what happens

In [15]:
eps = 0.1
cond = 'max_min'   # condition for agreement
k_avg = 10
M = 4

# auxiliary functions to parallelize computations
def get_temporal_results_one_run(N):
        
    # All below quantities are computed along time, then averaged over 'n_runs'
    nhe = []      # relative number of hyperedges
    s_avg = []    # average hyperedge size
    s_max = []    # max hyperedge size
    s_std = []    # std of hyperedge sizes
    n_agree = []
    n_merge = []
    n_split = []
    
    
    H = xgi.uniform_erdos_renyi_hypergraph(N, M, k_avg, p_type="degree")
    H.cleanup()
    groups_0 = H.edges.members()
    Model = HAD_model(eps, groups_0)
    results = Model.simulate(T=600, condition=cond)
    opinions = results['opinions']
    groups = results['groups']
    n_events = results['n_events']
    T = len(groups)

    final_ops = [opinions[n][-1] for n in H.nodes]
    # get results over time t
    for t in range(T):
        sizes_t = [len(he) for he in groups[t]]
        s_avg.append( np.average(sizes_t) )
        s_max.append( max(sizes_t) )
        s_std.append( np.std(sizes_t) )
        nhe.append( len(groups[t]) / len(groups_0) )
        
    res = {
        'nhe': nhe,
        's_avg': s_avg,
        's_max': s_max,
        's_std': s_std,
        'n_agree': n_events['agree'],
        'n_split': n_events['split'],
        'n_merge': n_events['merge'],
        'final_opinions': final_ops,
        'k_avg': k_avg
        }
    
    return res

In [17]:
%%time

Ns = [500, 1000, 2000, 5000]

p = Pool(processes=4)
results_ = p.map(get_temporal_results_one_run, Ns)

# save results
results = {Ns[i]: results_[i] for i in range(len(Ns))}

with open(f'../results/ER_{cond}/ER_M{M}_{cond}_epsilon_{eps}_temporal_one_run.pkl', 'wb') as fp:
        pickle.dump(results, fp)

CPU times: user 21 ms, sys: 31.2 ms, total: 52.2 ms
Wall time: 3min 6s
